# [IMG-I] Convoluties en Seam-Carving

Auteur: Brian van der Bijl (brian.vanderbijl@hu.nl)

In [ ]:
import numpy as np
import numpy.typing as npt
from typing import Tuple, Optional
from PIL import Image
from matplotlib import pyplot as plt
from ipywidgets import widgets, interact, fixed
from scipy import signal
from pylab import *

np.set_printoptions(precision=3, suppress=True)

centred_black = matplotlib.colors.LinearSegmentedColormap(
    'centred_black', 
    { 'red': ((0.0, 0.0, 1.0), (0.5, 0.0, 0.0), (1.0, 0.0, 0.0)),
      'green': ((0.0, 0.0, 0.0), (0.5, 0.0, 0.0), (1.0, 1.0, 1.0)),
      'blue': ((0.0, 0.0, 0.0), (0.5, 0.0, 0.0), (1.0, 1.0, 1.0))},
    256)

plt.style.use('dark_background')

def test_energy_map(fn):
    test_data = np.array([[0.1, 0.8, 0.8, 0.3, 0.5, 0.4],
                          [0.7, 0.8, 0.1, 0.0, 0.8, 0.4],
                          [0.8, 0.0, 0.4, 0.7, 0.2, 0.9],
                          [0.9, 0.0, 0.0, 0.5, 0.9, 0.4],
                          [0.2, 0.4, 0.0, 0.2, 0.4, 0.5],
                          [0.2, 0.4, 0.2, 0.5, 0.3, 0.0]])
    
    expected_res = np.array([[1.0, 1.1, 1.1, 0.6, 1.1, 1.7],
                             [0.9, 1.0, 0.3, 0.6, 1.7, 1.3],
                             [1.0, 0.2, 0.6, 0.9, 0.9, 1.7],
                             [1.3, 0.2, 0.2, 0.7, 1.3, 0.8],
                             [0.4, 0.6, 0.2, 0.4, 0.4, 0.5],
                             [0.2, 0.4, 0.2, 0.5, 0.3, 0.0]])
    
    expected_seam = np.array([3, 2, 1, 1, 2, 2])
    
    test_res, test_seam = fn(test_data)

    np.testing.assert_array_almost_equal_nulp(test_res, expected_res, 1)
    np.testing.assert_array_equal(test_seam, expected_seam)

def plot_annotated(img):
    fig, ax = plt.subplots()
    ax.imshow(img, interpolation="nearest", cmap="gray", vmax=img.max()*1.5)
    
    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            text = ax.text(j, i, f"{img[i, j]:.1f}",
                           ha="center", va="center", color="w")
            
    plt.show()

img1 = Image.open("images/persistence_memory.jpg")
img2 = Image.open("images/starry_night_rhone.jpg")
img3 = Image.open("images/lena.png")
img4 = Image.open("images/nakuno.jpg")

Kies een image file om mee te werken, zet deze globaal als `img_arr`.

In [ ]:
@interact(img=[("Persistence of Memory", img1), ("Starry Night over the Rhone", img2), ("Lena", img3), ("Nakuno", img4)])
def load_img(img):
    global img_arr, res
    img_arr = np.asarray(img)
    res = {img_arr.shape[1]: img_arr}
    
    plt.imshow(img_arr, interpolation="nearest")
    plt.show()

## Convoluties
Schrijf de volgende kernels als numpy arrays:

- $3 \times 3$ pixels
    - Sobel in (horizontaal) 
- $5 \times 5$ pixels
    - Box blur 
    - Gaussian blur ($5 \times 5$ pixels)
    - Unsharp mask

Maak ook een dictionary `kernels: dict[str, npt.NDArray]` om je kernels verderop makkelijker te kunnen testen:

```python
kernels = { "Box blur": box_blur, "Gauss (5)": gauss5, 
            "Unsharp mask (5)": unsharp5, "Sobel (X)": sobel_x, ... }
```

### Boven niveau
- Voeg zelf enkele zinvolle kernels toe
- Maak een verticale versie van Sobel om een verticale versie van het algoritme te ondersteunen *(dit is op zichzelf niet boven niveau)*

Voor het toepassen van de convoluties hebben we in de lessen zelf code geschreven. Omdat deze echter weinig geoptimaliseerd zal zijn, maken we hier gebruik van SciPy. Schrijf een wrapper `convolve_rgb(image: npt.NDArray, kernel: npt.NDArray) -> npt.NDArray` om `scipy.signal.convolve2D` een kernel per channel (R, G, B) over een afbeelding te laten convolueren. Het resultaat is een nieuwe afbeelding met drie kanalen.

In [ ]:
# TODO: Schrijf hier je code.

Voor de Sobel kernel willen we niet met RGB pixelwaardes rekenen, maar met de helderheid. Wij gebruiken hiervoor het gemiddelde van de R, G, en B channels. Maak een wrapper `convolve_brightness(image: npt.NDArray, kernel: npt.NDArray) -> npt.NDArray` om `scipy.signal` die een kernel over de brightness haalt. Het resultaat is een afbeelding met een enkel kanaal.

In [ ]:
# TODO: Schrijf hier je code.

Als alles goed is gegaan, kun je met deze cel je code testen.

In [ ]:
@interact(kernel_str=kernels.keys())
def demo(kernel_str: str):
    """Demonstrate a given convolution on the globally selected image."""
    brightness = ["Sobel (X)", "Sobel (Y)"]
    kernel = kernels[kernel_str]
    
    if kernel_str in brightness:
        res = convolve_brightness(img_arr, kernel)
    else:
        res = convolve_rgb(img_arr, kernel)
        
    plt.imshow(res, interpolation="nearest", cmap=centred_black)
    plt.show()

## Seam Carving
We gaan nu verder met het resultaat van Sobel op onze afbeelding. Pixels met een hoge absolute waarde beschouwen we als interessant, pixels met een lage absolute waarde beschouwen we als oninteressant. We willen de minst interessante seam vinden, zodat we deze kunnen verwijderen. Gegeven een array met getallen is een kandidaat seam een pad van pixels van de bovenste rij naar de onderste, waar steeds maximaal een pixel lateraal bewogen wordt (direct linksonder, direct onder, of direct rechtsonder). De beste seam, die we willen returnen, is de seam met de laagste som aan absolute pixelwaardes.

Om dit enigszins behapbaar te houden, gaan we een kaart bouwen. In pseudecode:
- Gegeven een `m, n` array `energy_map` met floats
- Maak `energy_map` positief door alle absolute waardes te nemen
- Maak een lege array `min_energy_map`
- Kopieer de onderste rij `m-1` van `energy_map` in `min_energy_map`
- Voor elke `rij` van `m-2` tot en met `0`:
    - Voor elke `kolom` `0` tot `n`: 
        - Kies uit de drie bereikbare pixels de laagste pixel-waarde als `min`
        - De waarde van pixel `rij, kolom` in `min_energy_map` wordt de waarde van pixel `rij, kolom` in `energy_map + min`
- Maak een lege array `seam_idxs` met lengte `m`
- Neem de index van de laagste waarde van rij `0` als `min` en zet deze als eerste waarde in `seam_idxs`
- Voor elke `rij` van `1` tot `m`:
    - Kies de pixel met de laagste waarde bereikbaar vanaf `min`, zet deze in `seam_idxs` en update `min`
- Geef de `min_energy_map` en `seam_idxs` terug

Schrijf de functie `minimal_energy_path(energy_map: npt.NDArray) -> Tuple[npt.NDArray, npt.NDArray]`.

**Hieronder staat een kleine voorbeeldmatrix om je code op te testen.** Deze vangt niet alles af, dus voel je vrij variaties toe te voegen om een beeld te krijgen hoe je code functioneert. Hou hierbij ook rekening met de letterlijke edge cases (pixels met minder dan 3 bereikbare cellen), de voorbeeldmatrix hieronder heeft geen seam langs de randen. 

### Boven Niveau
Hoewel we met dynamisch programmeren de search-space erg verkleind hebben ten opzichte van een brute-force search, is deze stap nog steeds relatief complex. Door alles met Python for-loops te schrijven ontstaat hier een bottleneck. Probeer `numpy` technieken zoals broadcasting toe te passen om je code te optimaliseren.

**NB:** begin met een naive versie en optimaliseer deze stap voor stap. Gooi geen oude versies weg totdat je zeker weet dat je optimalisatie functioneel identiek is.

In [ ]:
# TODO: Schrijf hier je code.

## Testmatrix
De matrix hieronder kun je gebruiken om je code te testen en te zien waar jouw resultaten afwijken van de verwachting:

In [ ]:
test = np.array([[0.1, 0.8, 0.8, 0.3, 0.5, 0.4],
                 [0.7, 0.8, 0.1, 0.0, 0.8, 0.4],
                 [0.8, 0.0, 0.4, 0.7, 0.2, 0.9],
                 [0.9, 0.0, 0.0, 0.5, 0.9, 0.4],
                 [0.2, 0.4, 0.0, 0.2, 0.4, 0.5],
                 [0.2, 0.4, 0.2, 0.5, 0.3, 0.0]])

plot_annotated(test)

Het resultaat zou er als volgt uit moeten zien:

In [ ]:
expected_res = np.array([[1.0, 1.1, 1.1, 0.6, 1.1, 1.7],
                             [0.9, 1.0, 0.3, 0.6, 1.7, 1.3],
                             [1.0, 0.2, 0.6, 0.9, 0.9, 1.7],
                             [1.3, 0.2, 0.2, 0.7, 1.3, 0.8],
                             [0.4, 0.6, 0.2, 0.4, 0.4, 0.5],
                             [0.2, 0.4, 0.2, 0.5, 0.3, 0.0]])

plot_annotated(expected_res)
print("Gevonden pad [3 2 1 1 2 2]")

Plot het resultaat van `minimal_energy_path`:

In [ ]:
test_res, test_seam = minimal_energy_path(test)

plot_annotated(test_res)
print("Gevonden pad", test_seam)

In [ ]:
test_energy_map(minimal_energy_path) # Sanity check

## Visualisatie
Zodra je de functie werkend hebt gekregen, kun je onderstaande cellen gebruiken om de toepassing op de afbeelding te visualiseren.

In [ ]:
@interact(seam=True, img=fixed(img_arr))
def show_energy_map(img: npt.NDArray, seam: bool = True):
    """Calculate and plot the energy map for a given image."""
    res = convolve_brightness(img, sobel_x)
    c, m = minimal_energy_path(res)

    if seam:
        for row in range(m.shape[0]):
            c[row, m[row]] = -c.max()
        
    plt.imshow(c, interpolation="nearest", cmap=centred_black, vmin=-c.max(), vmax=c.max())
    plt.show()

In [ ]:
@interact(seam=True, img=fixed(img_arr))
def show_seam(img: npt.NDArray, seam: bool = True):
    """Plot the seam on top of the original image."""
    res = convolve_brightness(img, sobel_x)
    c, m = minimal_energy_path(res)
    img = img.copy()

    if seam:
        for row in range(m.shape[0]):
            img[row, m[row]] = [255, 0, 0]
        
    plt.imshow(img, interpolation='nearest')
    plt.show()

## Bringing it together
Schrijf een functie `carve(img: npt.NDArray, tar: Optional[int] = None)` die, gegeven een afbeelding en een gewenste breedte, de afbeelding verkleint:
- Bereken hoeveel pixels moeten worden verwijderd, noem dit `n`
- Voor n iteraties:
    - Bereken de minst interessante `seam` in `img`
    - Verwijder de pixels in de `seam` uit `img`
    - Schuif zo nodig pixels door
- Geef de gereduceerde `img` terug.

Verifieer hierbij de user input: het algoritme reduceert alleen, en het heeft geen zin naar $\le 0$ pixels te schalen. Interpreteer `None` als de origenele breedte van de afbeelding.

### Boven Niveau
- Ondanks de opimalisaties kan het even duren om alle iteraties uit te voeren, en wordt er een hoop herberekend. Pas memoisation toe om tussenliggende waardes op te slaan, zodat de slider in de demo-cel responsief wordt.
- Breid het algoritme (en de onderliggende functies) uit om ook verticaal te kunnen carven. Gebruik een `bool` parameter om tussen beide modes te kiezen.
- Breid het algoritme uit zodat een afbeelding ook vergroot kan worden.
    - Waar loop je tegenaan?
    - Wat zou je kunnen doen om dit op te lossen?

In [ ]:
# TODO: Schrijf hier je code.

### Resultaat
Als alles werkt kun je hier interactief een afbeelding schalen:

In [ ]:
@interact(px = widgets.IntSlider(min=0, max=img_arr.shape[1], step=1, value=img_arr.shape[1]))
def main(px: int):
    red = carve(img_arr, px)
    
    _, ax = plt.subplots(2, 1, figsize=(10,10))
    
    ax[0].imshow(img_arr, interpolation='nearest', aspect='equal')
    ax[1].imshow(red, interpolation='nearest', aspect='equal')
    plt.show()

## Extra: Profiling information
Een van de mogelijkheden om de opdracht boven niveau te doen, is om je code te optimaliseren. Om de bottlenecks te vinden kun je gebruik maken van profiling:

In [ ]:
%load_ext line_profiler

In [ ]:
%lprun -f carve carve(img_arr, 100)